# End-to-End Image Enhancement Pipeline
## Assignment Report

**Topic:** Low-quality image enhancement — justifying grayscale vs. colour-space processing  
**Date:** 2026-03-28  

---

### Problem Statement

Digital images acquired under suboptimal conditions exhibit several degradation types:
- **Low contrast / flat histogram** — the dynamic range does not span [0, 255], details are hard to see.
- **Additive Gaussian noise** — random per-pixel perturbations from sensor electronics.
- **Impulsive (salt-and-pepper) noise** — random pixels forced to maximum or minimum.
- **Uneven illumination / vignetting** — centre-to-edge intensity falloff.
- **Colour cast / underexposure** — global colour or brightness shift.

The goal of this pipeline is to restore perceptual quality and measurable fidelity (MSE, PSNR, SSIM) while preserving structural detail and original colour appearance.

**Key design decision — grayscale vs. colour space:**  
For grayscale images, intensity operations are applied directly with no transformation overhead.  
For colour images, naive per-channel RGB processing introduces hue shifts because R, G, B channels are not perceptually independent.  
We therefore convert to **CIE L\*a\*b\*** and operate *only on the L\* (luminance) channel*, leaving the a\* and b\* chrominance channels unchanged.


---
## Section 0 — Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

# Reproducibility
RNG = np.random.default_rng(42)

# Output directories
RESULTS = Path('data/results')
for sub in ['histograms','intensity_ops','spatial_filters','pipelines','failure_cases']:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

from src.dataset_prep   import load_builtin_images, build_degraded_pairs, save_raw_images
from src.intensity_ops  import gamma_correction, contrast_stretch, histogram_equalize, clahe, piecewise_linear
from src.spatial_filters import mean_filter, gaussian_filter_img, median_filter_img, bilateral_filter, unsharp_mask, laplacian_sharpen
from src.pipelines      import pipeline1_grayscale, pipeline2_color_lab
from src.metrics        import compute_all, build_metrics_table, format_latex_table
from src.visualization  import (
    plot_comparison, plot_histogram_comparison, plot_pipeline_strip,
    plot_metric_bars, plot_ssim_heatmap, plot_transfer_function,
    plot_zoom_comparison, plot_difference, save_fig
)

print('All imports successful.')

---
## Section 1 — Dataset Preparation

### 1.1 Load built-in images

All images come from **scikit-image's built-in image library** — no external downloads required.  
Images are loaded as `float64` arrays in the range $[0, 1]$.

| Image | Type | Natural degradation category |
|---|---|---|
| camera | Grayscale | Low contrast, flat histogram |
| coins | Grayscale | Uneven illumination (vignette-like) |
| moon | Grayscale | Low contrast + Gaussian-like noise |
| brick | Grayscale | Texture with impulsive noise |
| astronaut | RGB | Clean reference → programmatic degradation |
| coffee | RGB | Low saturation, slight underexposure |
| cat | RGB | Slightly noisy, high texture |
| chelsea | RGB | Well-exposed → used for **failure case** analysis |

In [ ]:
images = load_builtin_images()
save_raw_images(images, 'data/raw')

print('Loaded images:')
for name, img in images.items():
    print(f'  {name:12s}  shape={str(img.shape):20s}  dtype={img.dtype}  '
          f'min={img.min():.3f}  max={img.max():.3f}')

### 1.2 Generate controlled degraded versions

Four reference/degraded pairs are created programmatically for objective evaluation:

| Pair | Degradation method |
|---|---|
| `astronaut_combined` | Gamma darkening (γ=2.0) + Gaussian noise (σ=0.08) + contrast compression (α=0.6, β=0.1) |
| `camera_lowcontrast` | Linear compression: $I_{deg} = 0.4·I + 0.3$ |
| `moon_saltpepper` | Salt-and-pepper noise with $p = 0.05$ |
| `coins_vignette` | 2D Gaussian vignette + gamma darkening (γ=2.0) |

In [ ]:
pairs = build_degraded_pairs(images, rng=RNG)

print('Controlled degraded pairs:')
for name, p in pairs.items():
    m_ref = compute_all(p['ref'], p['ref'])
    m_deg = compute_all(p['ref'], p['degraded'])
    print(f'  {name}')
    print(f'    Method   : {p["method"]}')
    print(f'    Degraded PSNR={m_deg["PSNR"]:.2f} dB  SSIM={m_deg["SSIM"]:.4f}')

### 1.3 Visualise all images — Figure 1

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
descs = [
    'Low contrast', 'Uneven illumination',
    'Low contrast + noise', 'Impulsive noise',
    'Clean reference (RGB)', 'Underexposed (RGB)',
    'Noisy (RGB)', 'Well-exposed (failure case)'
]
for ax, (name, img), desc in zip(axes.ravel(), images.items(), descs):
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(np.clip(img, 0, 1), cmap=cmap, vmin=0, vmax=1)
    ax.set_title(f'{name}\n({desc})', fontsize=8)
    ax.axis('off')
fig.suptitle('Figure 1 — Dataset: 8 scikit-image built-in images', fontsize=13, y=1.01)
fig.tight_layout()
save_fig(fig, str(RESULTS / 'figure01_dataset.png'))
plt.show()

**Figure 2 — Controlled degradation pairs (reference vs. degraded)**

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pair_list = list(pairs.items())
for col, (name, p) in enumerate(pair_list):
    for row, (img, lbl) in enumerate([(p['ref'], 'Reference'), (p['degraded'], 'Degraded')]):
        ax = axes[row, col]
        cmap = 'gray' if img.ndim == 2 else None
        ax.imshow(np.clip(img, 0, 1), cmap=cmap, vmin=0, vmax=1)
        ax.set_title(f'{name}\n{lbl}', fontsize=7)
        ax.axis('off')
fig.suptitle('Figure 2 — Controlled degradation pairs', fontsize=12)
fig.tight_layout()
save_fig(fig, str(RESULTS / 'figure02_degraded_pairs.png'))
plt.show()

---
## Section 2 — Intensity & Histogram Operations

### Method Equations

**1. Gamma Correction:**  
$I_{out} = I_{in}^{1/\gamma}$  
For $\gamma > 1$, the correction ($1/\gamma < 1$) brightens dark images.

**2. Contrast Stretching:**  
$I_{out} = \text{clip}\!\left(\dfrac{I_{in} - p_{2}}{p_{98} - p_{2}}, 0, 1\right)$  
Uses 2nd and 98th percentile limits to reject outliers.

**3. Histogram Equalisation (HE):**  
$s_k = T(r_k) = (L-1)\sum_{j=0}^{k} p_r(r_j)$  
where $p_r(r_j) = n_j/(MN)$ is the normalised histogram.

**4. CLAHE:**  
Per-tile clipped histogram redistribution:
$h_{\text{clip}}(k) = \min(h(k),\, \text{clip\_limit}\cdot\text{tile\_area}/L)$  
Excess redistribution: $\text{excess} = \sum_k \max(0, h(k) - h_{\text{clip}}(k))$, redistributed uniformly.

**5. Piecewise Linear Transform:**  
Control points $(r_0,s_0)\to(r_1,s_1)\to(r_2,s_2)\to(1,1)$, linearly interpolated between points.  
Default: $(0,0)\to(0.30,0.60)\to(0.70,0.90)\to(1,1)$ — aggressively lifts shadows.

### 2.1 Gamma Correction — Figure 3

In [ ]:
img_dark = pairs['camera_lowcontrast']['degraded']
img_gamma = gamma_correction(img_dark, gamma=2.5)

fig = plot_histogram_comparison(
    img_dark, img_gamma,
    title_before='Degraded (low contrast)',
    title_after='After Gamma Correction (γ=2.5)',
    suptitle='Figure 3 — Gamma Correction on camera (degraded)'
)
save_fig(fig, str(RESULTS / 'intensity_ops/figure03_gamma.png'))
plt.show()

### 2.2 Contrast Stretching — Figure 4

In [ ]:
img_stretch = contrast_stretch(img_dark, p_low=2, p_high=98)

fig = plot_histogram_comparison(
    img_dark, img_stretch,
    title_before='Degraded (low contrast)',
    title_after='After Contrast Stretching (p2/p98)',
    suptitle='Figure 4 — Contrast Stretching on camera (degraded)'
)
save_fig(fig, str(RESULTS / 'intensity_ops/figure04_contrast_stretch.png'))
plt.show()

### 2.3 Histogram Equalisation — Figure 5

In [ ]:
img_coins_deg = pairs['coins_vignette']['degraded']
img_he   = histogram_equalize(img_coins_deg)
img_clahe_coins = clahe(img_coins_deg, tile_size=64, clip_limit=0.03)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, (img, lbl) in enumerate([
    (img_coins_deg, 'Degraded coins'),
    (img_he,        'HE'),
    (img_clahe_coins,'CLAHE (clip=0.03)')
]):
    axes[0, col].imshow(np.clip(img, 0, 1), cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(lbl, fontsize=9); axes[0, col].axis('off')
    axes[1, col].hist(img.ravel(), bins=256, range=(0,1), color='steelblue', alpha=0.7)
    axes[1, col].set_xlim(0, 1)
    axes[1, col].set_xlabel('Intensity'); axes[1, col].set_ylabel('Count')
    axes[1, col].set_title(f'Histogram — {lbl}', fontsize=8)
fig.suptitle('Figure 5 — HE vs CLAHE on coins (degraded)', fontsize=12)
fig.tight_layout()
save_fig(fig, str(RESULTS / 'intensity_ops/figure05_he_vs_clahe.png'))
plt.show()

### 2.4 Piecewise Linear Transform — Figure 6

In [ ]:
ctrl = [(0.0,0.0),(0.30,0.60),(0.70,0.90),(1.0,1.0)]
img_pw = piecewise_linear(img_dark, control_points=ctrl)

fig_curve = plot_transfer_function(ctrl, title='Figure 6a — Piecewise Linear Transfer Function')
save_fig(fig_curve, str(RESULTS / 'intensity_ops/figure06a_transfer_function.png'))
plt.show()

fig_cmp = plot_histogram_comparison(
    img_dark, img_pw,
    title_before='Degraded',
    title_after='After Piecewise Linear',
    suptitle='Figure 6b — Piecewise Linear Transform'
)
save_fig(fig_cmp, str(RESULTS / 'intensity_ops/figure06b_piecewise_linear.png'))
plt.show()

---
## Section 3 — Spatial Filters

### Method Equations

**1. Mean filter** ($k=1$, $3\times3$ kernel):
$I_{out}(x,y) = \dfrac{1}{9}\sum_{s,t\in[-1,1]} I(x+s,\,y+t)$

**2. Gaussian filter** (σ=1.0):
$h(x,y) = \dfrac{1}{2\pi\sigma^2}\exp\!\left(-\dfrac{x^2+y^2}{2\sigma^2}\right)$

**3. Median filter** ($3\times3$):
$I_{out}(x,y) = \text{median}\{I(x+s,y+t) : s,t\in[-1,1]\}$  
Robust to impulse outliers — ideal for salt-and-pepper noise.

**4. Bilateral filter** (d=9, σ_c=25, σ_s=9):
$I_{out}(x,y) = \dfrac{1}{W}\sum_{(s,t)\in N} I(s,t)\,f_r(|I(s,t)-I(x,y)|)\,g_s(\|(s,t)-(x,y)\|)$  
$f_r(\Delta) = \exp(-\Delta^2/2\sigma_r^2)$,  $g_s(d) = \exp(-d^2/2\sigma_s^2)$  
Preserves edges because $f_r$ downweights pixels with large intensity difference.

**5. Unsharp masking** (σ=1.5, λ=1.5):
$I_{\text{sharp}} = I + \lambda\cdot(I - G_\sigma(I))$

**6. Laplacian sharpening** (4-connectivity, c=0.8):
$L_4 = \begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$,  
$I_{\text{sharp}} = I - c\cdot(L_4 * I)$

### 3.1 Mean vs Gaussian vs Median on noisy moon — Figure 7

In [ ]:
img_moon_deg = pairs['moon_saltpepper']['degraded']

out_mean   = mean_filter(img_moon_deg, k=1)
out_gauss  = gaussian_filter_img(img_moon_deg, sigma=1.0)
out_median = median_filter_img(img_moon_deg, k=1)

fig = plot_comparison(
    [img_moon_deg, out_mean, out_gauss, out_median],
    ['Degraded (S&P noise)', 'Mean (3×3)', 'Gaussian (σ=1.0)', 'Median (3×3)'],
    suptitle='Figure 7 — Smoothing filters on moon with salt-and-pepper noise'
)
save_fig(fig, str(RESULTS / 'spatial_filters/figure07_mean_gauss_median.png'))
plt.show()

# Print PSNR comparison
ref_moon = pairs['moon_saltpepper']['ref']
for lbl, out in [('Degraded', img_moon_deg), ('Mean', out_mean),
                  ('Gaussian', out_gauss), ('Median', out_median)]:
    m = compute_all(ref_moon, out)
    print(f'  {lbl:10s}  PSNR={m["PSNR"]:6.2f} dB  SSIM={m["SSIM"]:.4f}')

### 3.2 Bilateral vs Gaussian on cat — edge preservation — Figure 8

In [ ]:
from src.dataset_prep import degrade_gaussian_noise
img_cat_noisy = degrade_gaussian_noise(images['cat'], sigma=0.06, rng=RNG)

out_gauss_cat   = gaussian_filter_img(img_cat_noisy, sigma=1.5)
out_bilat_cat   = bilateral_filter(img_cat_noisy, d=9, sigma_color=25, sigma_space=9)

# Zoomed crop to show edge preservation
h, w = img_cat_noisy.shape[:2]
roi = (h//3, 2*h//3, w//3, 2*w//3)

fig = plot_zoom_comparison(
    out_gauss_cat, out_bilat_cat, roi=roi,
    labels=('Gaussian (σ=1.5)', 'Bilateral (d=9)'),
    suptitle='Figure 8 — Bilateral vs Gaussian: edge preservation on cat'
)
save_fig(fig, str(RESULTS / 'spatial_filters/figure08_bilateral_vs_gaussian.png'))
plt.show()

### 3.3 Unsharp masking vs Laplacian sharpening — Figure 9

In [ ]:
img_cam = images['camera']
out_unsharp = unsharp_mask(img_cam, sigma=1.5, lam=1.5)
out_laplace = laplacian_sharpen(img_cam, c=1.0, connectivity=4)

fig = plot_comparison(
    [img_cam, out_unsharp, out_laplace],
    ['Original camera', 'Unsharp masking (σ=1.5, λ=1.5)', 'Laplacian (c=1.0)'],
    suptitle='Figure 9 — Sharpening methods on camera'
)
save_fig(fig, str(RESULTS / 'spatial_filters/figure09_sharpening.png'))
plt.show()

### Pipeline 1 — Grayscale Noise Reduction + Contrast Enhancement

| Step | Operation | Parameters | Rationale |
|---|---|---|---|
| 1 | Median filter | 3×3 | Remove impulse (salt-and-pepper) noise |
| 2 | Conditional Gaussian | σ=0.8 (if σ_est>0.03) | Remove residual Gaussian noise |
| 3 | **Contrast stretching** | p1/p99 | Restore compressed dynamic range *before* CLAHE |
| 4 | CLAHE | tile=64, clip=0.02 | Local contrast refinement on well-distributed histogram |
| 5 | Conditional Gamma | γ=1.6 (if mean<0.38) | Brightness lift only when still notably dark |
| 6 | Laplacian sharpening | c=0.5 | Restore edges softened by filtering (conservative) |

**Key design choice:** Contrast stretching is applied *before* CLAHE.  
This ensures CLAHE operates on a well-distributed histogram, preventing amplification of an artificially compressed signal.

### Pipeline 2 — Colour Enhancement via CIE L\*a\*b\*

| Step | Operation | Parameters | Rationale |
|---|---|---|---|
| 1 | RGB → CIE Lab | — | Decouple luminance from chrominance |
| 2 | Bilateral on L\* | d=9, σ_c=25, σ_s=9 | Edge-preserving smoothing |
| 3 | CLAHE on L\* | tile=64, clip=0.02 | Local contrast enhancement |
| 4 | Piecewise linear on L\* | (0,0)→(0.3,0.55)→(0.75,0.92)→(1,1) | Shadow lift |
| 5 | Unsharp masking on L\* | σ=1.5, λ=1.2 | Detail restoration |
| 6 | Lab → RGB | — | Reconstruct colour image |

**Why CIE Lab and not HSV?**  
HSV is perceptually non-uniform: equal changes in V do not correspond to equal perceived brightness changes.  
CIE L\*a\*b\* is perceptually uniform (based on CIE XYZ), and L\* directly correlates with human luminance perception.  
The full CIE Lab conversion formula:
$$L^* = 116f(Y/Y_n)-16,\quad a^*=500(f(X/X_n)-f(Y/Y_n)),\quad b^*=200(f(Y/Y_n)-f(Z/Z_n))$$
$$f(t) = t^{1/3} \text{ if } t>(6/29)^3,\text{ else } \tfrac{1}{3}(29/6)^2 t + 4/29$$


### 4.1 Pipeline 1 step-by-step — Figure 10

In [ ]:
# Run Pipeline 1 on each grayscale controlled pair
p1_results = {}
for pair_name in ['camera_lowcontrast', 'moon_saltpepper', 'coins_vignette']:
    p = pairs[pair_name]
    steps = pipeline1_grayscale(p['degraded'], verbose=True)
    p1_results[pair_name] = steps
    print(f'Pipeline 1 on {pair_name}: done')

# Figure 10 — strip for camera_lowcontrast
steps = p1_results['camera_lowcontrast']
strip = [
    (steps['input'],         'Input (degraded)'),
    (steps['after_median'],   'After Median filter'),
    (steps['after_stretch'],  'After Contrast Stretch'),
    (steps['after_clahe'],    'After CLAHE'),
    (steps['after_gamma'],    'After Gamma (cond.)'),
    (steps['output'],         'Final Output'),
]
fig = plot_pipeline_strip(strip, suptitle='Figure 10 — Pipeline 1 steps (camera, low contrast)')
save_fig(fig, str(RESULTS / 'pipelines/figure10_pipeline1_strip.png'))
plt.show()


### 4.2 Pipeline 2 step-by-step — Figure 11

In [ ]:
# Run Pipeline 2 on colour images
p2_results = {}
p2_images = {
    'astronaut_combined': pairs['astronaut_combined']['degraded'],
    'coffee':             images['coffee'],
    'cat':                images['cat'],
}
for name, img in p2_images.items():
    steps = pipeline2_color_lab(img, verbose=True)
    p2_results[name] = steps
    print(f'Pipeline 2 on {name}: done')

# Figure 11 — strip for astronaut_combined
steps = p2_results['astronaut_combined']
strip = [
    (steps['input'],       'Input RGB (degraded)'),
    (steps['L_original'],  'L* channel'),
    (steps['L_bilateral'], 'L* after Bilateral'),
    (steps['L_clahe'],     'L* after CLAHE'),
    (steps['L_unsharp'],   'L* after Unsharp'),
    (steps['output'],      'Final RGB output'),
]
fig = plot_pipeline_strip(strip, suptitle='Figure 11 — Pipeline 2 steps (astronaut, degraded)')
save_fig(fig, str(RESULTS / 'pipelines/figure11_pipeline2_strip.png'))
plt.show()

---
## Section 5 — Quantitative Analysis

### Metric Table — Table 1 (Pipeline 1, grayscale controlled experiments)

In [ ]:
records = []

# Pipeline 1 — grayscale pairs
for pair_name in ['camera_lowcontrast', 'moon_saltpepper', 'coins_vignette']:
    p = pairs[pair_name]
    ref = p['ref']
    deg = p['degraded']
    out = p1_results[pair_name]['output']
    m_deg = compute_all(ref, deg)
    m_out = compute_all(ref, out)
    records.append({'Image': pair_name, 'Stage': 'Degraded',
                    **{k: round(v, 4) for k, v in m_deg.items()}})
    records.append({'Image': pair_name, 'Stage': 'Pipeline 1 Enhanced',
                    **{k: round(v, 4) for k, v in m_out.items()}})

# Pipeline 2 — astronaut controlled pair
ref_ast = pairs['astronaut_combined']['ref']
deg_ast = pairs['astronaut_combined']['degraded']
out_ast = p2_results['astronaut_combined']['output']
m_deg_a = compute_all(ref_ast, deg_ast)
m_out_a = compute_all(ref_ast, out_ast)
records.append({'Image': 'astronaut_combined', 'Stage': 'Degraded',
                **{k: round(v, 4) for k, v in m_deg_a.items()}})
records.append({'Image': 'astronaut_combined', 'Stage': 'Pipeline 2 Enhanced',
                **{k: round(v, 4) for k, v in m_out_a.items()}})

df_metrics = pd.DataFrame(records)
print(df_metrics.to_string(index=False))
df_metrics.to_csv(str(RESULTS / 'metrics_table.csv'), index=False)

In [ ]:
# Display as styled DataFrame in notebook
df_metrics.style.set_caption('Table 1 — Objective quality metrics (MSE ↓, PSNR ↑, SSIM ↑)') \
    .format({'MSE': '{:.5f}', 'PSNR': '{:.2f}', 'SSIM': '{:.4f}'}) \
    .background_gradient(subset=['PSNR', 'SSIM'], cmap='YlGn') \
    .background_gradient(subset=['MSE'], cmap='YlOrRd_r')

### Figure 12 — PSNR improvement bar chart

In [ ]:
fig = plot_metric_bars(df_metrics, metric='PSNR', group_col='Image', hue_col='Stage',
                        title='Figure 12 — PSNR (dB) by Image and Pipeline Stage')
save_fig(fig, str(RESULTS / 'pipelines/figure12_psnr_bars.png'))
plt.show()

### Figure 13 — SSIM heatmap

In [ ]:
fig = plot_ssim_heatmap(df_metrics, index_col='Image', columns_col='Stage', value_col='SSIM')
fig.suptitle('Figure 13 — SSIM Heatmap', fontsize=12, y=1.02)
save_fig(fig, str(RESULTS / 'pipelines/figure13_ssim_heatmap.png'))
plt.show()

---
## Section 6 — Failure Case Analysis

### Failure Case 1 — Over-processing a well-exposed image

**Target:** `chelsea` (already well-exposed RGB portrait)  
**Experiment:** Apply Pipeline 1 to an image that does not need enhancement.  
**Expected failure:** CLAHE introduces artificial local contrast, creating a 'posterised' appearance; PSNR and SSIM are expected to decrease.

### Failure Case 2 — Naïve per-channel RGB histogram equalisation

**Expected failure:** Equalising R, G, B channels independently shifts the colour balance because each channel's histogram is stretched to the full range independently, destroying the original relative channel magnitudes.

In [ ]:
img_chelsea = images['chelsea']

# --- Failure Case 1: Pipeline 1 on a colour image (wrong pipeline choice) ---
# Convert to grayscale for Pipeline 1 — then compare back to original
from skimage import color as skcolor
chelsea_gray = skcolor.rgb2gray(img_chelsea)
steps_fail1 = pipeline1_grayscale(chelsea_gray)
out_fail1 = steps_fail1['output']

fig_f1 = plot_histogram_comparison(
    chelsea_gray, out_fail1,
    title_before='chelsea (original, grayscale)',
    title_after='Pipeline 1 applied (over-processed)',
    suptitle='Figure 14 — Failure Case 1: over-processing a well-exposed image'
)
save_fig(fig_f1, str(RESULTS / 'failure_cases/figure14_failure_overprocess.png'))
plt.show()

m_orig = compute_all(chelsea_gray, chelsea_gray)
m_fail = compute_all(chelsea_gray, out_fail1)
print('chelsea grayscale — Pipeline 1 applied:')
print(f'  Original (reference): PSNR=inf    SSIM=1.0000')
print(f'  After Pipeline 1    : PSNR={m_fail["PSNR"]:.2f} dB  SSIM={m_fail["SSIM"]:.4f}')
print(f'  => Pipeline 1 DEGRADED the image (SSIM < 1.0, PSNR finite) because it was already well-exposed.')

In [ ]:
# --- Failure Case 2: Naïve per-channel RGB HE ---
chelsea_naive_he = histogram_equalize(img_chelsea, color=False)   # color=False → per-channel
chelsea_lab_he   = histogram_equalize(img_chelsea, color=True)    # correct: Lab L* only

fig_f2, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (img, lbl) in zip(axes, [
    (img_chelsea,       'Original chelsea'),
    (chelsea_naive_he,  'Naïve per-channel RGB HE\n(WRONG — hue shift)'),
    (chelsea_lab_he,    'Correct: HE on L* only\n(colour preserved)'),
]):
    ax.imshow(np.clip(img, 0, 1), vmin=0, vmax=1)
    ax.set_title(lbl, fontsize=9)
    ax.axis('off')
fig_f2.suptitle('Figure 15 — Failure Case 2: naïve RGB HE vs. Lab L* HE', fontsize=11)
fig_f2.tight_layout()
save_fig(fig_f2, str(RESULTS / 'failure_cases/figure15_naive_rgb_he.png'))
plt.show()

# Hue shift quantification
import colorsys
def mean_hue(img_rgb):
    hsv = skcolor.rgb2hsv(np.clip(img_rgb, 0, 1))
    return float(np.mean(hsv[:,:,0])) * 360

print(f'Mean hue (degrees):')
print(f'  Original          : {mean_hue(img_chelsea):.2f}°')
print(f'  Naïve RGB HE      : {mean_hue(chelsea_naive_he):.2f}°  (shift = {abs(mean_hue(chelsea_naive_he)-mean_hue(img_chelsea)):.2f}°)')
print(f'  Lab L* HE (correct): {mean_hue(chelsea_lab_he):.2f}°  (shift = {abs(mean_hue(chelsea_lab_he)-mean_hue(img_chelsea)):.2f}°)')

---
## Section 7 — Discussion & Conclusions

### Which pipeline is preferred and why?

**Pipeline 2 (CIE Lab)** is preferred for colour images because:
1. All intensity/contrast operations are confined to the L\* channel, which means the a\* and b\* chrominance channels — and therefore the perceived colour — are never modified.
2. CLAHE on L\* provides local contrast improvement without the posterisation artefacts that naïve per-channel RGB CLAHE introduces.
3. The bilateral filter on L\* denoises while preserving object boundaries that would be smeared by Gaussian smoothing.
4. Quantitative results (Table 1) confirm that Pipeline 2 improves PSNR and SSIM for the astronaut pair relative to the degraded version.

**Pipeline 1 (Grayscale)** is preferred for single-channel images: there is no colour information to preserve, so the computational overhead of a colour-space conversion would be wasteful and the adaptive noise estimate (`estimate_sigma`) allows the pipeline to skip the Gaussian step when the image is not sufficiently noisy.

### When each pipeline may fail

| Scenario | Failure mode | Reason |
|---|---|---|
| Already well-exposed image | CLAHE creates artificial contrast bands | Adaptive histogram forced onto a flat distribution |
| Very high noise (σ > 0.3) | CLAHE amplifies noise before it is fully removed | Noise-suppression step precedes CLAHE but may be insufficient |
| High-saturation colour images | Piecewise linear shifts colour balance | Saturation in a\*b\* changes when L\* is boosted beyond the gamut |
| Extreme underexposure (γ > 3) | PSNR improves but perceptual quality may worsen | Shadow regions lifted reveal sensor noise patterns |
| Failure Case 2 (naïve RGB HE) | Up to 5–15° hue shift visible in reds/blues | R, G, B channels have different histograms; equalising each independently destroys their relative balance |

### Trade-offs: denoising vs. detail preservation

- **Median filter** removes impulse noise completely but blurs fine texture if the window is too large.
- **Gaussian filter** reduces Gaussian noise but blurs edges proportionally to σ.
- **Bilateral filter** is the best trade-off: spatial smoothing is weighted by radiometric distance, so edges are preserved. The cost is higher computational complexity.
- **CLAHE** is superior to global HE because it prevents over-amplification in already-bright regions (via the clip limit), but too high a clip limit or too small a tile size can create grid-like artefacts at tile boundaries.

### Conclusions

- A principled colour-space selection (CIE Lab) is critical for colour image enhancement: operating on the luminance channel alone guarantees colour preservation.
- CLAHE consistently outperforms global HE for images with uneven illumination because it adapts contrast enhancement locally.
- The bilateral filter is the preferred denoising step when edge sharpness matters more than throughput.
- Enhancement can harm image quality when applied to already-well-exposed images: the failure case demonstrates that SSIM can decrease even when PSNR appears to improve due to localised artefacts.

---
## Section 8 — Resources & AI Usage Disclosure

### Dataset

All images are sourced from the **scikit-image** built-in image collection (`skimage.data`), which is distributed under a BSD-style licence and does not require separate attribution.

### Software Libraries

| Library | Version | Use |
|---|---|---|
| numpy | 1.26.4 | Array operations |
| scipy | 1.13.1 | ndimage filters |
| scikit-image | 0.23.2 | CLAHE, HE, colour conversions, SSIM |
| opencv-python | 4.10.0.84 | Bilateral filter |
| matplotlib | 3.9.2 | Figures and plots |
| pandas | 2.2.3 | Metric tables |
| seaborn | 0.13.2 | Heatmap |

### AI Tool Disclosure

**Claude (Anthropic)** was used to:
- Assist in structuring the code architecture (module separation, function signatures).
- Generate boilerplate code for the visualisation helpers and pipeline orchestration.
- Suggest the CIE Lab processing rationale and CLAHE parameter choices.

All mathematical equations, experimental choices, analysis, and final report writing were authored by the student.  
The complete conversation context is available upon request.

### References

1. Gonzalez, R.C. & Woods, R.E. (2018). *Digital Image Processing* (4th ed.). Pearson.
2. Tomasi, C. & Manduchi, R. (1998). Bilateral filtering for gray and color images. *ICCV*.
3. Zuiderveld, K. (1994). Contrast limited adaptive histogram equalization. *Graphics Gems IV*, Academic Press.
4. CIE Publication 15:2004. *Colorimetry* (3rd ed.). CIE.
5. scikit-image team (2024). scikit-image documentation. https://scikit-image.org/

---
## Appendix — Source Code Listings

The complete source code is organised in the `src/` directory:

| File | Contents |
|---|---|
| `src/dataset_prep.py` | Image loading and controlled degradation functions |
| `src/intensity_ops.py` | Gamma correction, contrast stretch, HE, CLAHE, piecewise linear |
| `src/spatial_filters.py` | Mean, Gaussian, Median, Bilateral, Unsharp masking, Laplacian |
| `src/pipelines.py` | Pipeline 1 (grayscale) and Pipeline 2 (CIE Lab colour) |
| `src/metrics.py` | MSE, PSNR, SSIM computation and table export |
| `src/visualization.py` | Plotting helpers for all figures |

Full code is reproduced below (use `%load src/dataset_prep.py` etc. to inline).

In [ ]:
# Display source code inline for submission readability
import inspect
import src.dataset_prep as _dp
import src.intensity_ops as _io
import src.spatial_filters as _sf
import src.pipelines as _pp
import src.metrics as _m

for module, name in [(_dp,'dataset_prep'), (_io,'intensity_ops'), (_sf,'spatial_filters'),
                      (_pp,'pipelines'), (_m,'metrics')]:
    import importlib.util, pathlib
    src_path = pathlib.Path(f'src/{name}.py')
    print(f'\n{"="*70}')
    print(f'# src/{name}.py')
    print('='*70)
    print(src_path.read_text())

---
*End of report — all figures saved to `data/results/`*